In [20]:
from langchain_community.document_loaders import TextLoader, CSVLoader
from pygments.lexer import words

In [21]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.bbc.co.uk/news")
data = loader.load()

In [22]:
import os
os.environ["USER_AGENT"] = "MyRAGApp/1.0"

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.bbc.co.uk/news")
data = loader.load()

In [72]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(r"C:\Users\mr\Downloads\movies _data.csv")
data = loader.load()

print(f"Total rows loaded: {len(data)}")
print(data[0])  # preview first row

Total rows loaded: 38
page_content='movie_id: 101
title: K.G.F: Chapter 2
industry: Bollywood
release_year: 2022
imdb_rating: 8.4
studio: Hombale Films
language_id: 3' metadata={'source': 'C:\\Users\\mr\\Downloads\\movies _data.csv', 'row': 0}


In [24]:
len(data)

38

In [25]:
data[2].page_content

'movie_id: 103\ntitle: Thor: The Dark World\nindustry: Hollywood\nrelease_year: 2013\nimdb_rating: 6.8\nstudio: Marvel Studios\nlanguage_id: 5'

In [26]:
data[2].metadata

{'source': 'C:\\Users\\mr\\Downloads\\movies _data.csv', 'row': 2}

In [27]:
!pip install unstructured libmagic python-magic python-magic-bin


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
from langchain_community.document_loaders import UnstructuredURLLoader

In [49]:
import requests
from langchain_core.documents import Document  # ← changed from langchain.schema
from bs4 import BeautifulSoup

urls = [
    "https://www.moneycontrol.com/news/business/banks/hdfc-bank-re-appoints-sanmoy-chakrabarti-as-chief-risk-officer-11259771.html",
    "https://www.moneycontrol.com/news/business/markets/market-corrects-post-rbi-ups-inflation-forecast-icrr-bet-on-these-top-10-rate-sensitive-stocks-ideas-11142611.html"
]

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

data = []
for url in urls:
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator="\n", strip=True)
    data.append(Document(page_content=text, metadata={"source": url}))

print(len(data))
print(data[0].page_content[:500])

2
HDFC Bank re-appoints Sanmoy Chakrabarti as Chief Risk Officer
English
Hindi
Gujarati
Specials
Search Quotes, News, Mutual Fund NAVs
Trending Stocks
HDFC Bank
INE040A01034, HDFCBANK, 500180
Reliance
INE002A01018, RELIANCE, 500325
Ola Electric
INE0LXG01040, OLAELEC, 544225
Vedanta
INE205A01025, VEDL, 500295
Larsen
INE018A01030, LT, 500510
Quotes
Mutual Funds
Commodities
Futures & Options
Unlisted Shares
Currency
News
Topic
Cryptocurrency
Forum
Notices
Videos
Glossary
All
My Alerts
Go Ad-Free
Hell


In [50]:
data[0].metadata

{'source': 'https://www.moneycontrol.com/news/business/banks/hdfc-bank-re-appoints-sanmoy-chakrabarti-as-chief-risk-officer-11259771.html'}

In [51]:
text = """Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is embroiled in a catastrophic blight and famine, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for humankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007 and was originally set to be directed by Steven Spielberg.
Kip Thorne, a Caltech theoretical physicist and 2017 Nobel laureate in Physics,[4] was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm. Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects, and the company Double Negative created additional digital effects.

Interstellar premiered in Los Angeles on October 26, 2014. In the United States, it was first released on film stock, expanding to venues using digital projectors. The film received generally positive reviews from critics and grossed over $677 million worldwide ($715 million after subsequent re-releases), making it the tenth-highest-grossing film of 2014.
It has been praised by astronomers for its scientific accuracy and portrayal of theoretical astrophysics.[5][6][7] Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades."""

In [53]:
text[:200]

'Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.\nIt stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt'

In [54]:
# Step 1: Extract all text from your loaded documents
full_text = " ".join([doc.page_content for doc in data])

# Step 2: Split into words
words = full_text.split()

# Step 3: Your chunking logic (with a small fix — flush remaining text at the end)
chunks = []
s = ""
for word in words:
    s += " " + word
    if len(s) >= 200:
        chunks.append(s.strip())
        s = ""  # reset after appending

if s:  # don't lose the last chunk if it's under 200 chars
    chunks.append(s.strip())

print(f"Total chunks: {len(chunks)}")
print(chunks[0])

Total chunks: 107
HDFC Bank re-appoints Sanmoy Chakrabarti as Chief Risk Officer English Hindi Gujarati Specials Search Quotes, News, Mutual Fund NAVs Trending Stocks HDFC Bank INE040A01034, HDFCBANK, 500180 Reliance INE002A01018,


In [55]:
chunks

['HDFC Bank re-appoints Sanmoy Chakrabarti as Chief Risk Officer English Hindi Gujarati Specials Search Quotes, News, Mutual Fund NAVs Trending Stocks HDFC Bank INE040A01034, HDFCBANK, 500180 Reliance INE002A01018,',
 'RELIANCE, 500325 Ola Electric INE0LXG01040, OLAELEC, 544225 Vedanta INE205A01025, VEDL, 500295 Larsen INE018A01030, LT, 500510 Quotes Mutual Funds Commodities Futures & Options Unlisted Shares Currency',
 'News Topic Cryptocurrency Forum Notices Videos Glossary All My Alerts Go Ad-Free Hello, Login Hello, Login Log-in or Sign-Up My Account My Profile My Portfolio My Watchlist My Alerts My Messages Price',
 'Alerts My Profile My PRO My Portfolio My Watchlist My Alerts My Messages Price Alerts Logout Loans up to ₹50 LAKHS Fixed Deposits Credit Cards Lifetime Free Credit Score Loan against MFs Chat with Us',
 'Download App Follow us on: ->-> Go PRO Now PRO PRO Markets HOME INDIAN INDICES STOCK ACTION All Stats Top Gainers Top Losers Only Buyers Only Sellers 52 Week High 52 

In [56]:
from langchain_text_splitters import CharacterTextSplitter  # capital C and T and S

splitter = CharacterTextSplitter(
    separator="\n",       # not a method call — it's a parameter
    chunk_size=200,       # not chunks_size
    chunk_overlap=0       # not chunks_overlap
)

chunks = splitter.split_documents(data)  # pass your loaded docs, not raw text
print(len(chunks))
print(chunks[0].page_content)

Created a chunk of size 399, which is longer than the specified 200
Created a chunk of size 250, which is longer than the specified 200
Created a chunk of size 388, which is longer than the specified 200
Created a chunk of size 244, which is longer than the specified 200
Created a chunk of size 399, which is longer than the specified 200
Created a chunk of size 250, which is longer than the specified 200
Created a chunk of size 388, which is longer than the specified 200
Created a chunk of size 244, which is longer than the specified 200
Created a chunk of size 220, which is longer than the specified 200


122
HDFC Bank re-appoints Sanmoy Chakrabarti as Chief Risk Officer
English
Hindi
Gujarati
Specials
Search Quotes, News, Mutual Fund NAVs
Trending Stocks
HDFC Bank
INE040A01034, HDFCBANK, 500180
Reliance


In [57]:
for chunk in chunks:
    print(chunk)

page_content='HDFC Bank re-appoints Sanmoy Chakrabarti as Chief Risk Officer
English
Hindi
Gujarati
Specials
Search Quotes, News, Mutual Fund NAVs
Trending Stocks
HDFC Bank
INE040A01034, HDFCBANK, 500180
Reliance' metadata={'source': 'https://www.moneycontrol.com/news/business/banks/hdfc-bank-re-appoints-sanmoy-chakrabarti-as-chief-risk-officer-11259771.html'}
page_content='INE002A01018, RELIANCE, 500325
Ola Electric
INE0LXG01040, OLAELEC, 544225
Vedanta
INE205A01025, VEDL, 500295
Larsen
INE018A01030, LT, 500510
Quotes
Mutual Funds
Commodities
Futures & Options' metadata={'source': 'https://www.moneycontrol.com/news/business/banks/hdfc-bank-re-appoints-sanmoy-chakrabarti-as-chief-risk-officer-11259771.html'}
page_content='Unlisted Shares
Currency
News
Topic
Cryptocurrency
Forum
Notices
Videos
Glossary
All
My Alerts
Go Ad-Free
Hello, Login
Hello, Login
Log-in
or
Sign-Up
My Account
My Profile
My Portfolio
My Watchlist' metadata={'source': 'https://www.moneycontrol.com/news/business/banks

In [58]:
from langchain_text_splitters import RecursiveCharacterTextSplitter  # ← fix is here

r_splitter = RecursiveCharacterTextSplitter(
    separators=["\\n\\n", "\\n", " "],
    chunk_size=200,
    chunk_overlap=0,
    length_function=len
)

chunks = r_splitter.split_documents(data)
print(len(chunks))

117


In [59]:
for chunk in chunks:
    print(chunk)

page_content='HDFC Bank re-appoints Sanmoy Chakrabarti as Chief Risk Officer
English
Hindi
Gujarati
Specials
Search Quotes, News, Mutual Fund NAVs
Trending Stocks
HDFC Bank
INE040A01034, HDFCBANK,' metadata={'source': 'https://www.moneycontrol.com/news/business/banks/hdfc-bank-re-appoints-sanmoy-chakrabarti-as-chief-risk-officer-11259771.html'}
page_content='500180
Reliance
INE002A01018, RELIANCE, 500325
Ola Electric
INE0LXG01040, OLAELEC, 544225
Vedanta
INE205A01025, VEDL, 500295
Larsen
INE018A01030, LT, 500510
Quotes
Mutual Funds
Commodities
Futures &' metadata={'source': 'https://www.moneycontrol.com/news/business/banks/hdfc-bank-re-appoints-sanmoy-chakrabarti-as-chief-risk-officer-11259771.html'}
page_content='Options
Unlisted Shares
Currency
News
Topic
Cryptocurrency
Forum
Notices
Videos
Glossary
All
My Alerts
Go Ad-Free
Hello, Login
Hello, Login
Log-in
or
Sign-Up
My Account
My Profile
My Portfolio
My' metadata={'source': 'https://www.moneycontrol.com/news/business/banks/hdfc-bank

In [63]:
text_chunks = text.split("\n\n")  # plain strings, not Document objects
for chunk in text_chunks:
    print(len(chunk))

437
716
611


In [64]:
first_split = text_chunks[0].split("\n")
print(len(first_split))

3


In [65]:
second_split = text_chunks[1].split("\n")
print(len(second_split))

4


In [66]:
second_split.pop()

'Interstellar uses extensive practical and miniature effects, and the company Double Negative created additional digital effects.'

In [67]:
first_split = text.split("\n\n")[0]
first_split

'Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.\nIt stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.\nSet in a dystopian future where humanity is embroiled in a catastrophic blight and famine, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for humankind.'

In [68]:
len(first_split)

437

In [69]:
second_split = first_split.split("\n")
second_split

['Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.',
 'It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.',
 'Set in a dystopian future where humanity is embroiled in a catastrophic blight and famine, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for humankind.']

In [70]:
for split in second_split:
    print(len(split))

105
120
210


In [71]:
second_split[2]

'Set in a dystopian future where humanity is embroiled in a catastrophic blight and famine, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for humankind.'